# Data Preparation

## Pilot Study Data Preparation

In [25]:
import os
import glob
import librosa
import soundfile as sf
import numpy as np
import pandas as pd
import random
from pydub import AudioSegment

# Define paths relative to the notebook workspace
DATA_DIR = "../data/synthesized"
MIXED_DIR = "../data/mixed"

INSTRUMENTS = os.listdir(DATA_DIR)
INSTRUMENT_SYNONYMS = {
    "baglama_saz": ["bağlama", "saz", "baglama", "turkish saz", "bağlama saz"],
    "kanun": ["kanun", "qanun", "turkish zither"],
    "kemence": ["kemençe", "kemence", "black sea fiddle", "karadeniz kemençesi"],
    "ney": ["ney", "turkish flute", "reed flute"],
    "ud": ["ud", "oud", "turkish lute"],
    "zurna": ["zurna", "turkish oboe"]
}

os.makedirs(MIXED_DIR, exist_ok=True)

print("Directories setup complete.")

Directories setup complete.


In [19]:
SR = 22050 # Standard sample rate
CHUNK_LENGTH = 5 # seconds
SAMPLES_PER_CHUNK = SR * CHUNK_LENGTH

use this probability distribution:

70% of your dataset: Mix exactly 2 random instruments. (This builds your strong foundational weights).

25% of your dataset: Mix exactly 3 random instruments. (This teaches the model to handle some complexity).

5% of your dataset: Mix 4 random instruments. (This makes the model robust for real-world YouTube recordings).

0%: Do not bother mixing 5 or 6 simultaneously for this pilot study.

In [27]:
def get_random_5s_chunk(audio_segment, chunk_length_ms=5000):
    """
    Extracts a random 5-second slice from a full audio file.
    Pads with silence if the original file is too short.
    """
    audio_length = len(audio_segment) # pydub measures length in milliseconds

    # Edge Case: The song is shorter than 5 seconds
    if audio_length <= chunk_length_ms:
        # Pad the end with pure silence to guarantee a perfect 5.000s tensor
        padding = AudioSegment.silent(duration=(chunk_length_ms - audio_length))
        return audio_segment + padding

    # Standard Case: The song is longer than 5 seconds
    # Pick a random start time that leaves exactly enough room for the 5s chunk
    max_start_time = audio_length - chunk_length_ms
    start_time = random.randint(0, max_start_time)
    end_time = start_time + chunk_length_ms

    # Slice the audio exactly like a Python list
    return audio_segment[start_time:end_time], start_time, end_time

In [38]:
list(range(2-1))

[0]

In [ ]:
def generate_training_tuple(tuple_id):
    """
    Generates a single training tuple (Mixture, Target, Prompt) following
    the strict curriculum learning probability distribution.
    """

    # 1. THE CURRICULUM DISTRIBUTION
    num_instruments = random.choices([2, 3, 4], weights=[0.70, 0.25, 0.05], k=1)[0]

    # 2. Pick the unique instruments for this specific mixture
    selected_instruments = random.sample(INSTRUMENTS, num_instruments)

    # 3. Choose one of those selected instruments to be the "Ground Truth Target"
    target_instrument = random.choice(selected_instruments)

    mixed_audio = None
    target_audio = None

    # for storing selected audio start and end
    # example for num_instrument == 2; {0: {}, 'target': {}}
    selected_instrument_info = {}
    non_target_idx = 0

    for inst in selected_instruments:
        inst_dir = os.path.join(DATA_DIR, inst)
        available_files = [f for f in os.listdir(inst_dir) if f.endswith('.wav')]

        # ASYNCHRONOUS MIXING: Pick a random file to prevent the 'Perfect Unison' trap
        random_file = random.choice(available_files)
        file_path = os.path.join(inst_dir, random_file)

        # Load the clean audio stem
        full_audio = AudioSegment.from_wav(file_path)

        # crop to 5-second stem
        stem, start_time, end_time = get_random_5s_chunk(full_audio)


        # Add the audio to the complex mixture
        if mixed_audio is None:
            mixed_audio = stem
        else:
            mixed_audio = mixed_audio.overlay(stem)

        inst_metadata = {
            "instrument": inst,
            "song": random_file,
            "start_time": start_time,
            "end_time": end_time
        }

        # If this is the target instrument, save this pure stem as the ground truth
        if inst == target_instrument:
            target_audio = stem
            selected_instrument_info['target'] = inst_metadata
        else:
            # Save non-target instruments under 0, 1, 2, etc.
            # Using str(non_target_idx) ensures the JSON keys are strings natively
            selected_instrument_info[str(non_target_idx)] = inst_metadata
            non_target_idx += 1

    # 4. Save the generated .wav files
    mixture_path = os.path.join(MIXED_DIR, f"mix_{tuple_id:04d}_{num_instruments}inst.wav")
    target_path = os.path.join(MIXED_DIR, f"target_{tuple_id:04d}_{target_instrument}.wav")

    mixed_audio.export(mixture_path, format="wav")
    target_audio.export(target_path, format="wav")

    # 5. Format the Text Prompt for the SAM Model
    prompt = random.choice(INSTRUMENT_SYNONYMS[target_instrument])

    return {
        "prompt": prompt,
        "mixture_path": mixture_path,
        "target_path": target_path,
        "selected_instrument_info": selected_instrument_info
    }

In [ ]:
from tqdm.auto import tqdm
import json

def build_dataset(num_samples=720):
    """
    Executes the generation loop to build the pilot dataset.
    720 samples equals roughly 1 hour of audio (if chunks are 5 seconds).
    """
    print(f"Initializing generation of {num_samples} training tuples...")
    dataset_metadata = []

    for i in tqdm(range(num_samples), desc="Generating tuples"):
        metadata = generate_training_tuple(i)
        dataset_metadata.append(metadata)

    # Save the mapping file for the fine-tuning loop
    jsonl_path = os.path.join('..', 'data', "mixed_metadata.jsonl")
    with open(jsonl_path, 'w', encoding='utf-8') as f:
        for entry in dataset_metadata:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    print(f"Dataset complete! Metadata mapped and saved to {jsonl_path}")

/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [33]:
# generate an initial pilot batch of 720 tuples
build_dataset(720)

Initializing generation of 720 training tuples...


Generating tuples: 100%|██████████| 720/720 [01:18<00:00,  9.15it/s]

Dataset complete! Metadata mapped and saved to ../data/dataset.jsonl
